# MiniMax H3 — 静止画1枚から縦動画

G4 は生成のときだけ使う。重みのダウンロードは CPU ランタイム。

- Drive は **fireworker06@gmail.com**。重みは `マイドライブ/h3-weights/MiniMax-H3`。完成 mp4 は `マイドライブ/h3-runner/output/orbis01.mp4`。
- T2VA と Ref2VA を両方置いた合計は **210.3GB**（text_encoder 66.73 + transformer 66.28 + transformer_ref 66.28 + vae 10.42 + audio_vae 0.61 + 設定）。ダウンロード中は未完了シャード最大 5.1GB が足されて約 215.4GB。`FL2VA/` と `Ref2VA/`（各 144.1GB）は落とさない。
- `/content/drive` の空き表示は VM ディスクの値なので、そこで止めない。シャードを1本ずつ `/content/tmp_hf` に落として Drive へ移す。`/` の空きが 30GB を切ったら、drivefs のキャッシュが上がるまで待つ。G4 では落とさない。
- 生成は Drive を直接読む。CPU と G4 は別の VM なので、G4 のローカルへ先に置くことはできない。G4 上でコピーしてから読むと、Drive からの読みに書き込みと再読みが足される。読み込み後の offload はホスト RAM に置く。
- 15秒を1本では出さない。6秒（文章だけ）と 9秒（静止画）を作り、ffmpeg で 15秒にする。Ref2VA は 352×640 のまま。
- LoRA は要らない。`HF_TOKEN` はライセンス同意が要るときだけ Colab のシークレットに入れる。値は表示しない。

[Open in Colab](https://colab.research.google.com/github/fireworker011/Research/blob/cursor/h3-still-to-video-7cb8/h3-runner/minimax_h3_still.ipynb)


## A. CPU ランタイム

ランタイムのタイプを **CPU**（GPU なし）にする。上の導入セルの次から、A のセルだけ実行する。


### A. Drive

fireworker06@gmail.com でマウントする。空きの数字は参考。Colab のマウントは VM ディスクの値を返すので Drive 実容量ではない。


In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
WEIGHTS = Path("/content/drive/MyDrive/h3-weights")
OUT = Path("/content/drive/MyDrive/h3-runner/output")
WEIGHTS.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
os.environ["H3_HF_CACHE"] = str(WEIGHTS)
os.environ["H3_OUT_DIR"] = str(OUT)
os.environ["HF_HOME"] = "/content/tmp_hf/hf-home"

token = ""
try:
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    token = ""
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN をシークレットから読みました（値は表示しません）")
else:
    print("HF_TOKEN は未設定。401 のときだけ、ライセンス同意後にシークレットへ入れてこのセルからやり直す。")

usage = shutil.disk_usage("/content/drive")
root = shutil.disk_usage("/")
print(f"参考: /content/drive の statvfs 空き {usage.free / 1e9:.1f} GB / 全体 {usage.total / 1e9:.1f} GB（10^9）")
print("Colab のマウントは VM ディスクの値を返すので Drive 実容量ではない。この数字では止めない。")
print(f"ローカル / の空き {root.free / 1e9:.1f} GB。drivefs のキャッシュはここを使う。")
print("両方置く合計は 210.3GB。シャードを1本ずつ落とす。")
print("重み:", WEIGHTS)
print("完成 mp4:", OUT)
print("画面のアカウントが fireworker06@gmail.com であることを確認する。")


### A. リポジトリ

Research を `/content/Research` に clone する。ランナーは `/content/Research/h3-runner/run_h3.py`。シンボリックリンクは使わない。


In [ ]:
import subprocess
from pathlib import Path

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-still-to-video-7cb8"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

clone_runner()


### A. パッケージ

ダウンロードに `huggingface_hub` だけ入れる。torch は触らない。


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25.0"])
print("huggingface_hub ok")


### A. 重み準備

無ければシャードを1本ずつ Drive に落とす。サイズが一致するファイルは飛ばして続きからやる。GPU は使わない。

一時ファイルは `/content/tmp_hf`。1本ごとに Drive へ移してローカルを消し、`os.sync()` する。`drive.flush_and_unmount` は使わない。`/` の空きが 30GB を切ったらキャッシュが上がるまで待つ。

`--vram-gb 95 --host-ram-gb 176.9` は前回の G4 実測で、計画の文面用。このセルはディスクだけ見る。


In [ ]:
import os
import subprocess
import sys
import threading
from pathlib import Path

def run_logged(cmd):
    """Colab does not show a child process's own fds. Print them from this cell."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    stderr_lines = []

    def pump(stream, acc):
        for line in stream:
            print(line, end="", flush=True)
            if acc is not None:
                acc.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout, None))
    err_thread = threading.Thread(target=pump, args=(proc.stderr, stderr_lines))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    if code != 0:
        print(f"\n終了コード {code}。stderr 全文:", flush=True)
        print("".join(stderr_lines), flush=True)
        raise SystemExit(code)
    return code

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-still-to-video-7cb8"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

script = clone_runner()
cmd = [
    sys.executable, "-u", str(script),
    "--preset", "orbis01",
    "--cache-dir", os.environ["H3_HF_CACHE"],
    "--out-dir", os.environ["H3_OUT_DIR"],
    "--prepare-weights",
    "--vram-gb", "95",
    "--host-ram-gb", "176.9",
]
run_logged(cmd)
print("重み準備が終わった。ランタイムを G4 に変えて B へ進む。")


## B. G4 ランタイム

ランタイムのタイプを **G4 GPU**（RTX PRO 6000）にする。CPU とは別の VM なので、Drive とリポジトリはここでも取る。生成はこのあとの1セル。


### B. 準備

Drive、パッケージ、リポジトリ。`pip install -U torchao==0.18.0` で `FqnToConfig` を直す。Colab の torch は置き換えない。


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
WEIGHTS = Path("/content/drive/MyDrive/h3-weights")
OUT = Path("/content/drive/MyDrive/h3-runner/output")
if not WEIGHTS.is_dir():
    raise SystemExit(f"重みフォルダが無い: {WEIGHTS}。先に CPU の A を実行する。")
OUT.mkdir(parents=True, exist_ok=True)
os.environ["H3_HF_CACHE"] = str(WEIGHTS)
os.environ["H3_OUT_DIR"] = str(OUT)
os.environ["HF_HOME"] = str(WEIGHTS / "hf-home")

token = ""
try:
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    token = ""
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF_TOKEN をシークレットから読みました（値は表示しません）")

usage = shutil.disk_usage("/content/drive")
print(f"Drive 空き {usage.free / 1e9:.1f} GB / 全体 {usage.total / 1e9:.1f} GB（10^9）")

import torch
if not torch.cuda.is_available():
    raise SystemExit("GPU がオフです。ランタイムを G4 にして、このセルからやり直す。")
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
mem = Path("/proc/meminfo").read_text(encoding="utf-8")
host = int(next(line.split()[1] for line in mem.splitlines() if line.startswith("MemTotal:"))) / 1024 ** 2
print(f"GPU: {props.name}  VRAM: {vram:.1f} GB  ホストRAM: {host:.1f} GB  torch {torch.__version__}")
os.environ["H3_VRAM_GB"] = f"{vram:.4f}"
os.environ["H3_HOST_RAM_GB"] = f"{host:.4f}"
if vram < 24:
    raise SystemExit("VRAM が 24GB 未満です。G4 に変えてやり直す。")

if shutil.which("ffmpeg") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])

pkgs = [
    "git+https://github.com/huggingface/diffusers.git@5ff8e59ff9fe81c6e2df4fb4c6ea0d97a5df5ab2",
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "safetensors>=0.4.3",
    "huggingface_hub>=0.25.0",
    "av>=11.0.0",
    "imageio>=2.34.0",
    "imageio-ffmpeg>=0.5.0",
    "soundfile>=0.12.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torchao==0.18.0"])
from torchao.quantization import FqnToConfig
print("torchao FqnToConfig ok", "ffmpeg", shutil.which("ffmpeg"))

def clone_runner():
    root = Path("/content/Research")
    branch = "cursor/h3-still-to-video-7cb8"
    repo = "https://github.com/fireworker011/Research.git"
    script = root / "h3-runner" / "run_h3.py"
    if not script.is_file():
        if root.exists():
            raise SystemExit(f"{root} があるが {script} が無い。このフォルダを消してやり直す。")
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", branch, repo, str(root)])
    else:
        subprocess.check_call(["git", "-C", str(root), "fetch", "--depth", "1", "origin", branch])
        subprocess.check_call(["git", "-C", str(root), "checkout", branch])
        subprocess.check_call(["git", "-C", str(root), "pull", "--ff-only", "origin", branch])
    if not script.is_file():
        raise SystemExit(f"ランナーが無い: {script}")
    print("runner", script)
    return script

clone_runner()


### B. 生成

この1セルが生成の最初から結合まで。ダウンロードしない。Drive の `h3-weights` を直接読む。

`orbis01_6s.mp4` が Drive にあれば T2VA をスキップする。切れたら、ランタイムが生きていればこのセルだけ、死んでいれば「B. 準備」からやり直す。


In [ ]:
import os
import subprocess
import sys
import threading
from pathlib import Path

def run_logged(cmd):
    """Colab does not show a child process's own fds. Print them from this cell."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    stderr_lines = []

    def pump(stream, acc):
        for line in stream:
            print(line, end="", flush=True)
            if acc is not None:
                acc.append(line)

    out_thread = threading.Thread(target=pump, args=(proc.stdout, None))
    err_thread = threading.Thread(target=pump, args=(proc.stderr, stderr_lines))
    out_thread.start()
    err_thread.start()
    code = proc.wait()
    out_thread.join()
    err_thread.join()
    if code != 0:
        print(f"\n終了コード {code}。stderr 全文:", flush=True)
        print("".join(stderr_lines), flush=True)
        raise SystemExit(code)
    return code

script = Path("/content/Research/h3-runner/run_h3.py")
if not script.is_file():
    raise SystemExit(f"ランナーが無い: {script}。B. 準備からやり直す。")
cmd = [
    sys.executable, "-u", str(script),
    "--preset", "orbis01",
    "--cache-dir", os.environ["H3_HF_CACHE"],
    "--out-dir", os.environ["H3_OUT_DIR"],
    "--vram-gb", os.environ["H3_VRAM_GB"],
    "--host-ram-gb", os.environ["H3_HOST_RAM_GB"],
    "--seed", "0",
]
run_logged(cmd)
out = Path(os.environ["H3_OUT_DIR"]) / "orbis01.mp4"
if not out.is_file():
    raise SystemExit(f"完成 mp4 がありません: {out}")
print("完成:", out, "bytes", out.stat().st_size)
